# 01 - Förbered och utforska

Det här notebooket läser in verklig transaktionsdata, gör en join mot en lookup-tabell och bygger en kundnivåtabell för clustering.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists() and (ROOT.parent.parent / 'data').exists():
    ROOT = ROOT.parent.parent

FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

transactions = pd.read_csv(ROOT / 'data' / 'raw' / 'transactions.csv', parse_dates=['InvoiceDate'])
regions = pd.read_csv(ROOT / 'data' / 'raw' / 'regions.csv')
transactions.head()

In [ ]:
transactions.shape, regions.shape

In [ ]:
# Enkel datakontroll
quality = pd.DataFrame({
    'dtype': transactions.dtypes.astype(str),
    'missing_count': transactions.isna().sum(),
    'missing_pct': (transactions.isna().mean() * 100).round(2),
})
quality

In [ ]:
cleaned = transactions.dropna(subset=['CustomerID']).copy()
cleaned = cleaned[(cleaned['Quantity'] > 0) & (cleaned['UnitPrice'] > 0)].copy()
cleaned['CustomerID'] = cleaned['CustomerID'].astype(int)
cleaned['Country'] = cleaned['Country'].fillna('Unknown')
cleaned['Revenue'] = cleaned['Quantity'] * cleaned['UnitPrice']
cleaned['RegionGroup'] = cleaned['Country'].map(
    regions.set_index('Country')['RegionGroup'].to_dict()
).fillna('Other')
cleaned.head()

In [ ]:
invoice_level = (
    cleaned.groupby(['CustomerID', 'InvoiceNo'], as_index=False)
    .agg(invoice_total=('Revenue', 'sum'), invoice_date=('InvoiceDate', 'max'))
)
ref_date = cleaned['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = (
    cleaned.groupby('CustomerID', as_index=False)
    .agg(
        last_purchase_date=('InvoiceDate', 'max'),
        frequency=('InvoiceNo', 'nunique'),
        monetary=('Revenue', 'sum'),
        basket_size=('Quantity', 'sum'),
        unique_products=('StockCode', 'nunique'),
        avg_unit_price=('UnitPrice', 'mean'),
        Country=('Country', 'first'),
        RegionGroup=('RegionGroup', 'first'),
    )
)
customers = customers.merge(
    invoice_level.groupby('CustomerID', as_index=False).agg(avg_order_value=('invoice_total', 'mean')),
    on='CustomerID',
    how='left',
)
customers['recency_days'] = (ref_date - customers['last_purchase_date']).dt.days
customers['avg_items_per_invoice'] = customers['basket_size'] / customers['frequency']
customers = customers[
    ['CustomerID', 'Country', 'RegionGroup', 'recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price', 'last_purchase_date']
].sort_values('CustomerID').reset_index(drop=True)

customers.to_csv(ROOT / 'data' / 'raw' / 'customers.csv', index=False)
customers.to_csv(ROOT / 'data' / 'processed' / 'customer_enriched.csv', index=False)
customers.head()

In [ ]:
sns.set_theme(style='whitegrid', context='talk')
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=customers,
    x='recency_days',
    y='monetary',
    size='frequency',
    hue='RegionGroup',
    sizes=(20, 240),
    alpha=0.75,
    palette='Set2',
    ax=ax,
    legend=False,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.invert_xaxis()
ax.set_title('RFM-karta: recency mot omsättning')
ax.set_xlabel('Recency-dagar, lägre är bättre')
ax.set_ylabel('Omsättning, log-skala')
ax.text(0.02, 0.04, 'Punktstorlek = frekvens', transform=ax.transAxes, fontsize=11, color='#555555')

region_order = ['Storbritannien och Irland', 'Norden', 'Sydeuropa', 'Mellanöstern', 'Västeuropa', 'Övrigt', 'APAC', 'Afrika', 'Nordamerika', 'Östeuropa', 'Sydamerika', 'Brittiska öarna']
region_values = [value for value in region_order if value in customers['RegionGroup'].unique().tolist()]
region_palette = sns.color_palette('Set2', n_colors=len(region_values))
region_handles = [Line2D([0], [0], marker='o', color='w', label=label, markerfacecolor=color, markersize=10) for label, color in zip(region_values, region_palette)]
ax.legend(handles=region_handles, title='Regiongrupp', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
fig.savefig(FIGURES / 'rfm_scatter.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
kolumner = ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price']
plt.figure(figsize=(11, 8))
sns.heatmap(customers[kolumner].corr(numeric_only=True), annot=True, fmt='.2f', cmap='Blues', center=0)
plt.title('Korrelation mellan kundfeatures')
plt.show()

Notebookens output är en kundnivåtabell i `data/processed/customer_enriched.csv`. Notebook 2 använder samma schema för träning och inference.